# `graep.logging.setup_logging` — verbosity walkthrough

`setup_logging` is the single entry point for configuring logging in the framework. It is **idempotent**: call it again from any cell to change verbosity at runtime.

Verbosity scale (low = quiet, high = loud):

| Value | Level    | Meaning                                |
|------:|----------|----------------------------------------|
|     0 | CRITICAL | only catastrophic failures             |
|     1 | ERROR    | errors only                            |
|     2 | WARNING  | warnings + errors                      |
|     3 | INFO     | informational messages (default)       |
|     4 | DEBUG    | everything                             |

In [1]:
import logging

from graep.logging import setup_logging

demo = logging.getLogger("graep.demo")

## Default — verbosity 3 (INFO)

DEBUG messages are filtered out; INFO and above are shown.

In [2]:
setup_logging(3)

demo.debug("this should NOT appear")
demo.info("hello at INFO")
demo.warning("warning visible")
demo.error("error visible")

[INFO:graep.demo:<module>:L.4] hello at INFO
[WARNING:graep.demo:<module>:L.5] warning visible
[ERROR:graep.demo:<module>:L.6] error visible


## Bump to verbosity 4 (DEBUG)

Calling `setup_logging` again wipes the previous handler and re-installs one with the new level. No accumulation, no duplicate lines.

In [3]:
setup_logging(4)

demo.debug("now visible at DEBUG")
demo.info("still visible")

[DEBUG:graep.demo:<module>:L.3] now visible at DEBUG
[INFO:graep.demo:<module>:L.4] still visible


## Per-module overrides

Keep root at DEBUG, but mute one chatty module to WARNING.

In [4]:
setup_logging(4, per_module={"graep.noisy": 2})

noisy = logging.getLogger("graep.noisy")
demo.debug("graep.demo at DEBUG — visible")
noisy.debug("graep.noisy at DEBUG — should NOT appear")
noisy.info("graep.noisy at INFO — should NOT appear")
noisy.warning("graep.noisy at WARNING — visible")

[DEBUG:graep.demo:<module>:L.4] graep.demo at DEBUG — visible
[WARNING:graep.noisy:<module>:L.7] graep.noisy at WARNING — visible


## Resetting verbosity

Per-module overrides do not leak across `setup_logging` calls. The next call re-establishes a clean state.

In [5]:
setup_logging(1)  # errors only

demo.warning("warning suppressed")
demo.error("only errors visible now")
noisy.warning("graep.noisy override is gone — also suppressed")

[ERROR:graep.demo:<module>:L.4] only errors visible now


## Invalid verbosity

Out-of-range values raise `ValueError` immediately.

In [6]:
try:
    setup_logging(7)
except ValueError as e:
    print(f"got expected error: {e}")

got expected error: verbosity must be in 0..4, got 7
